# Compensation Mechanisms Study — Phase 3 (FP32 & QAT INT8)

**Author:** Rafael  
**Dataset:** Tiny-ImageNet-200 (200 classes, 64×64 RGB)  
**Goal:** Train and compare all Phase 3 compensation mechanisms in FP32, then apply
Quantization-Aware Training (QAT) to the QAT-compatible models to obtain INT8 versions.
Report **Top-1** and **Top-5** accuracy for every model.

| Model | Mechanism | QAT | Notes |
|---|---|---|---|
| AlexNetBottleneck | 1×1→3×3→1×1 bottleneck | ✓ | Conv-BN-ReLU triples; nested blocks |
| AlexNetFactorized | 1×3 + 3×1 asymmetric | ✓ | Inception-style spatial factorization |
| AlexNetGroupConv | Grouped convolutions (g=4) | ✓ | 4× fewer cross-channel params |
| AlexNetDepthwiseSep | Depthwise separable | ✓ | MobileNet-style DW+PW pairs |
| AlexNetResidual | Residual skip connections | ✓ | FloatFunctional add; most impactful |
| AlexNetFire | Fire modules (SqueezeNet) | ✓ | torch.cat; nested squeeze+expand |
| AlexNetGAP | Global Average Pooling head | ✓ | Identical backbone to AlexNet3x3 |
| AlexNetSE | Squeeze-and-Excitation | ✗ | Sigmoid not fbgemm-compatible |

## Notebook layout

1. Imports & reproducibility
2. Dataset & loaders
3. Model shape check
4. Model registration (fuse maps)
5. FP32 training
6. QAT training (all except AlexNetSE)
7. INT8 conversion & CPU evaluation
8. FP32 evaluation (reload best checkpoints)
9. Final comparison table
10. Persist experiment summary

## 1. Imports & reproducibility

In [ ]:
import json
import os
import random
import sys
from dataclasses import asdict, replace
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torchinfo
import wandb

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from ml import (
    DataConfig, TrainerConfig, QATConfig,
    create_imagenet_loaders,
    MODEL_REGISTRY, register_model,
    Trainer, make_qat_callback,
    load_best_model, convert_to_int8,
    find_fuse_groups,
    auto_resume_path,
    create_results_summary, disk_mb,
    compute_flops, make_run_summary,
)
from configs.loader import load_config

from models import (
    AlexNetTV_bottleneck,   AlexNetBottleneck,
    AlexNetTV_factorized,   AlexNetFactorized,
    AlexNetTV_groupconv,    AlexNetGroupConv,
    AlexNetTV_depthwisesep, AlexNetDepthwiseSep,
    AlexNetTV_residual,     AlexNetResidual,
    AlexNetTV_fire,         AlexNetFire,
    AlexNetTV_gap,          AlexNetGAP,
    AlexNetTV_se,           AlexNetSE,
)

torch.backends.quantized.engine = "fbgemm"

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_DIR    = project_root / "checkpoints" / "compensation_phase3"
RESULTS_DIR = project_root / "results" / "compensation_phase3"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

data_cfg = DataConfig(**load_config("data.yaml"))
data_cfg.seed = SEED
fp32_cfg = TrainerConfig(**load_config("training.yaml"))
qat_cfg  = QATConfig(**load_config("qat.yaml"))
print(device)

## 2. Dataset & loaders

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")
train_path = os.path.join(dataset_path, "tiny-imagenet-200", "train")
data_cfg.dataset_path = train_path
print("Tiny-ImageNet train path:", train_path)

train_ds, val_ds, train_loader, val_loader = create_imagenet_loaders(data_cfg)

print(f"Train samples: {len(train_ds):,}")
print(f"Val   samples: {len(val_ds):,}")
print(f"Classes:       {data_cfg.num_classes}")
print(f"Batches/epoch: train={len(train_loader)}, val={len(val_loader)}")

## 3. Model shape check

Eight Phase 3 compensation mechanisms — all from `models/compensation.py`:

| Name | Mechanism | BN? | Head |
|---|---|---|---|
| AlexNetBottleneck | 1×1→3×3→1×1 bottleneck | Yes | Linear(256, 200) |
| AlexNetFactorized | 1×3 + 3×1 factorized | Yes | 3-layer MLP |
| AlexNetGroupConv | Grouped conv (g=4) | Yes | 3-layer MLP |
| AlexNetDepthwiseSep | Depthwise separable | Yes | Linear(256, 200) |
| AlexNetResidual | Residual connections | Yes | 3-layer MLP |
| AlexNetFire | Fire modules | Yes | Linear(256, 200) |
| AlexNetGAP | Global Average Pooling | No | Linear(256, 200) |
| AlexNetSE | Squeeze-and-Excitation | No | 3-layer MLP |

In [ ]:
x = torch.randn(2, 3, data_cfg.img_size, data_cfg.img_size)
for label, ctor in [
    ("AlexNetBottleneck",   AlexNetTV_bottleneck),
    ("AlexNetFactorized",   AlexNetTV_factorized),
    ("AlexNetGroupConv",    AlexNetTV_groupconv),
    ("AlexNetDepthwiseSep", AlexNetTV_depthwisesep),
    ("AlexNetResidual",     AlexNetTV_residual),
    ("AlexNetFire",         AlexNetTV_fire),
    ("AlexNetGAP",          AlexNetTV_gap),
    ("AlexNetSE",           AlexNetTV_se),
]:
    m = ctor().eval()
    with torch.no_grad():
        y = m(x)
    assert y.shape == (2, data_cfg.num_classes), f"{label}: unexpected shape {y.shape}"
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    print(f"{label:22s} OK -> {tuple(y.shape)}, params={info.total_params/1e6:.2f}M")

## 4. Model registration

Fuse maps for flat-Sequential models are hand-written; nested block models
(Bottleneck, Residual, Fire) use `find_fuse_groups()` for auto-detection.
`AlexNetSE` is excluded from QAT — Sigmoid in SE blocks is not fbgemm-compatible.

In [ ]:
# ── Nested block models — auto-detect Conv-BN(-ReLU) groups ──────────────────
FUSE_BOTTLENECK = find_fuse_groups(AlexNetTV_bottleneck())
FUSE_RESIDUAL   = find_fuse_groups(AlexNetTV_residual())
FUSE_FIRE       = find_fuse_groups(AlexNetTV_fire())

# ── AlexNetFactorized: 1×3→3×1 Conv-BN-ReLU pairs (10 groups) ───────────────
# Stage 1 has two MaxPools (indices 6,7) to compensate for no strided conv
FUSE_FACTORIZED = [
    ["0","1","2"],["3","4","5"],      # Stage 1 (MaxPool×2 at 6,7)
    ["8","9","10"],["11","12","13"],  # Stage 2 (MaxPool at 14)
    ["15","16","17"],["18","19","20"],# Stage 3
    ["21","22","23"],["24","25","26"],# Stage 4
    ["27","28","29"],["30","31","32"],# Stage 5
]

# ── AlexNetGroupConv: one Conv-BN-ReLU per stage (5 groups) ──────────────────
FUSE_GROUPCONV = [
    ["0","1","2"],   # Stage 1 (MaxPool at 3)
    ["4","5","6"],   # Stage 2 (MaxPool at 7)
    ["8","9","10"],  # Stage 3
    ["11","12","13"],# Stage 4
    ["14","15","16"],# Stage 5
]

# ── AlexNetDepthwiseSep: DW+PW Conv-BN-ReLU pairs (10 groups) ────────────────
FUSE_DEPTHWISESEP = [
    ["0","1","2"],["3","4","5"],      # Stage 1 (MaxPool at 6)
    ["7","8","9"],["10","11","12"],   # Stage 2 (MaxPool at 13)
    ["14","15","16"],["17","18","19"],# Stage 3
    ["20","21","22"],["23","24","25"],# Stage 4
    ["26","27","28"],["29","30","31"],# Stage 5
]

# ── AlexNetGAP: Conv-ReLU pairs, no BN (identical fuse map to AlexNet3x3) ────
FUSE_GAP = [["0","1"],["3","4"],["6","7"],["8","9"],["10","11"]]

MODEL_REGISTRY.clear()
register_model("alexnet_bottleneck",   AlexNetTV_bottleneck,   fuse_map=FUSE_BOTTLENECK,   lr=1e-3)
register_model("alexnet_factorized",   AlexNetTV_factorized,   fuse_map=FUSE_FACTORIZED,   fuse_root_attr="features", lr=3e-4)
register_model("alexnet_groupconv",    AlexNetTV_groupconv,    fuse_map=FUSE_GROUPCONV,    fuse_root_attr="features", lr=1e-3)
register_model("alexnet_depthwisesep", AlexNetTV_depthwisesep, fuse_map=FUSE_DEPTHWISESEP, fuse_root_attr="features", lr=1e-3)
register_model("alexnet_residual",     AlexNetTV_residual,     fuse_map=FUSE_RESIDUAL,     lr=3e-4)
register_model("alexnet_fire",         AlexNetTV_fire,         fuse_map=FUSE_FIRE,         lr=1e-3)
register_model("alexnet_gap",          AlexNetTV_gap,          fuse_map=FUSE_GAP,          fuse_root_attr="features", lr=3e-4)
register_model("alexnet_se",           AlexNetTV_se,           fuse_map=[],                lr=3e-4)  # ponytail: QAT skipped — Sigmoid not fbgemm-compatible

print("Registered:", list(MODEL_REGISTRY))

## 5. FP32 training

In [ ]:
fp32_training_results = {}

for name, spec in MODEL_REGISTRY.items():
    # Auto-detect resume checkpoint
    resume_from = auto_resume_path(SAVE_DIR, name)
    
    # Restore W&B run if resuming
    run_id = None
    if resume_from:
        meta_path = SAVE_DIR / f"{name}_meta.json"
        if meta_path.exists():
            run_id = json.loads(meta_path.read_text()).get("wandb_run_id")
    
    model_cfg = replace(fp32_cfg, lr=spec.get("lr", fp32_cfg.lr))
    print("=" * 72)
    print(f"Training: {name}  lr={model_cfg.lr}  epochs={model_cfg.epochs}")
    if resume_from:
        print(f"(Resuming from checkpoint)")
    print("=" * 72)

    model = spec["ctor"]().to(device)

    run = wandb.init(
        project="alexnet-phase3",
        group="fp32-phase3",
        name=f"{name}_fp32",
        id=run_id,
        resume="allow" if run_id else None,
        config={**asdict(model_cfg), "arch": name, "phase": "fp32",
                "num_classes": data_cfg.num_classes, "img_size": data_cfg.img_size,
                "dataset": "tiny-imagenet-200"},
        tags=["phase3", "compensation", "tiny-imagenet", "fp32"],
        mode="offline",
    )

    trainer = Trainer(
        model, train_loader, val_loader, model_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
        wandb_run=run,
        log_file=SAVE_DIR / f"{name}.log",
    )
    results = trainer.fit(resume_from=resume_from)
    wandb.finish()

    fp32_training_results[name] = results

    del model
    torch.cuda.empty_cache()

print("\nFP32 training complete.")

## 6. Quantization-Aware Training (QAT)

`AlexNetSE` is excluded: Sigmoid in SE blocks is not fbgemm-compatible.
All other models support QAT. QAT trains on GPU; `convert` and INT8 inference run on CPU (fbgemm).

In [ ]:
QAT_SKIP = {"alexnet_se"}  # ponytail: Sigmoid in SE not fbgemm-compatible

qat_train_cfg = replace(
    fp32_cfg,
    epochs=qat_cfg.epochs,
    lr=qat_cfg.lr,
    weight_decay=qat_cfg.weight_decay,
    use_amp=False,  # AMP incompatible with fake-quant observers
)

qat_models = {}
qat_training_results = {}

for name in MODEL_REGISTRY:
    if name in QAT_SKIP:
        print(f"Skipping QAT for {name} (QAT_SKIP).")
        continue

    # Auto-detect resume checkpoint
    resume_from = auto_resume_path(SAVE_DIR, f"qat_{name}")
    
    # Restore W&B run if resuming
    run_id = None
    if resume_from:
        meta_path = SAVE_DIR / f"qat_{name}_meta.json"
        if meta_path.exists():
            run_id = json.loads(meta_path.read_text()).get("wandb_run_id")

    print("=" * 72)
    print(f"QAT fine-tuning: {name}")
    if resume_from:
        print(f"(Resuming from checkpoint)")
    print("=" * 72)

    qat_model = build_qat(name, save_dir=SAVE_DIR, device=device)
    cb = make_qat_callback(qat_cfg.freeze_bn_epoch, qat_cfg.disable_observer_epoch)

    run = wandb.init(
        project="alexnet-phase3",
        group="qat-phase3",
        name=f"{name}_qat",
        id=run_id,
        resume="allow" if run_id else None,
        config={
            **asdict(qat_train_cfg),
            "arch": name,
            "phase": "qat",
            "freeze_bn_epoch": qat_cfg.freeze_bn_epoch,
            "disable_observer_epoch": qat_cfg.disable_observer_epoch,
            "num_classes": data_cfg.num_classes, "img_size": data_cfg.img_size,
            "dataset": "tiny-imagenet-200",
        },
        tags=["phase3", "compensation", "tiny-imagenet", "qat"],
        mode="offline",
    )

    trainer = Trainer(
        qat_model, train_loader, val_loader, qat_train_cfg,
        device, SAVE_DIR, f"qat_{name}", num_classes=data_cfg.num_classes,
        wandb_run=run,
        epoch_callback=cb,
        log_file=SAVE_DIR / f"qat_{name}.log",
    )
    results = trainer.fit(resume_from=resume_from)
    wandb.finish()

    qat_training_results[name] = results
    qat_models[name] = qat_model.cpu()

    del qat_model
    torch.cuda.empty_cache()

print("\nQAT training complete.")

## 7. INT8 conversion & CPU evaluation

`torch.ao.quantization.convert` materialises true quantised ops and must run on CPU.

In [ ]:
val_loader_cpu = torch.utils.data.DataLoader(
    val_ds, batch_size=data_cfg.batch_size, shuffle=False, num_workers=0, pin_memory=False,
)
cpu_device = torch.device("cpu")

int8_models = {name: convert_to_int8(m.cpu().eval()) for name, m in qat_models.items()}

for name, m in int8_models.items():
    torch.save(m.state_dict(), SAVE_DIR / f"{name}.pth")
print("INT8 conversion done.")

int8_metrics = {}
for name, m in int8_models.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(
        m.cpu().eval(), val_loader_cpu, val_loader_cpu, dummy_cfg,
        cpu_device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    metrics = trainer.evaluate(topk=(1, 5))
    int8_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

## 8. FP32 evaluation — reload best checkpoints

In [ ]:
CTORS = {
    "alexnet_bottleneck":       "alexnet_factorized":       "alexnet_groupconv":    "alexnet_depthwisesep":     "alexnet_residual":     "alexnet_fire":         "alexnet_gap":          "alexnet_se":       }

fp32_best = {name: load_best_model(name, ctor, SAVE_DIR, device) for name, ctor in CTORS.items()}

fp32_metrics = {}
for name, m in fp32_best.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    trainer = Trainer(
        m, train_loader, val_loader, dummy_cfg,
        device, SAVE_DIR, name, num_classes=data_cfg.num_classes,
    )
    metrics = trainer.evaluate(topk=(1, 5))
    fp32_metrics[name] = metrics
    print(f"{name:22s} | loss={metrics['loss']:.4f} | top1={metrics['top1']:.2f}% | top5={metrics['top5']:.2f}%")

## 9. Final comparison table

In [ ]:
rows = []

for name, m in fp32_best.items():
    info = torchinfo.summary(m, input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": name, "precision": "FP32",
        "top1_%": fp32_metrics[name]["top1"],
        "top5_%": fp32_metrics[name]["top5"],
        "loss":   fp32_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB":  disk_mb(SAVE_DIR / f"{name}_best.pth"),
    })

for name, m in int8_models.items():
    info = torchinfo.summary(fp32_best[name], input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    rows.append({
        "model": f"{name}_INT8", "precision": "INT8",
        "top1_%": int8_metrics[name]["top1"],
        "top5_%": int8_metrics[name]["top5"],
        "loss":   int8_metrics[name]["loss"],
        "params_M": info.total_params / 1e6,
        "size_MB":  disk_mb(SAVE_DIR / f"{name}.pth"),
    })

df = build_comparison_table(rows)
df.to_csv(RESULTS_DIR / "final_comparison.csv", index=False)
df

In [ ]:
import json as _json

# --- Benchmark FP32 models (GPU) ---
fp32_benchmarks = {}
for name, m in fp32_best.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    t = Trainer(m, train_loader, val_loader, dummy_cfg, device, SAVE_DIR, name,
                num_classes=data_cfg.num_classes)
    fp32_benchmarks[name] = t.benchmark()
    print(f"{name:22s} FP32 | {fp32_benchmarks[name]['latency_ms_per_image']:.3f} ms/img")

# --- Benchmark INT8 models (CPU) ---
int8_benchmarks = {}
for name, m in int8_models.items():
    dummy_cfg = replace(fp32_cfg, use_amp=False)
    t = Trainer(m.cpu().eval(), val_loader_cpu, val_loader_cpu, dummy_cfg, cpu_device, SAVE_DIR, name,
                num_classes=data_cfg.num_classes)
    int8_benchmarks[name] = t.benchmark(warmup=50)
    print(f"{name:22s} INT8 | {int8_benchmarks[name]['latency_ms_per_image']:.3f} ms/img")

# --- FLOPs ---
fp32_flops = {}
for name, m in fp32_best.items():
    fp32_flops[name] = compute_flops(m.cpu().eval())
    print(f"{name:22s} | {fp32_flops[name]['macs']/1e9:.3f} GMACs")

# --- Per-model summary JSONs ---
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for name in fp32_best:
    info = torchinfo.summary(fp32_best[name], input_size=(1, 3, data_cfg.img_size, data_cfg.img_size), verbose=0)
    summary = make_run_summary(
        name=name,
        mode="fp32+qat+int8",
        fit_results=fp32_training_results.get(name, {}),
        fp32_eval=fp32_metrics[name],
        params_m=info.total_params / 1e6,
        fp32_size_mb=disk_mb(SAVE_DIR / f"{name}_best.pth"),
        int8_size_mb=disk_mb(SAVE_DIR / f"{name}.pth"),
        fp32_benchmark=fp32_benchmarks[name],
        flops_results=fp32_flops[name],
        int8_eval=int8_metrics.get(name),
        int8_benchmark=int8_benchmarks.get(name),
    )
    out = RESULTS_DIR / f"{name}_summary.json"
    with open(out, "w") as f:
        _json.dump(summary, f, indent=2, default=str)
    print(f"Saved: {out.name}")

In [ ]:
print("=" * 72)
print("RANKING BY TOP-1 ACCURACY (all precisions)")
print("=" * 72)
ranked = df.sort_values("top1_%", ascending=False).reset_index(drop=True)
for i, row in ranked.iterrows():
    print(f"{i+1:2d}. {row['model']:26s} [{row['precision']}] | "
          f"top1={row['top1_%']:6.2f}% | top5={row['top5_%']:6.2f}% | "
          f"params={row['params_M']:6.2f}M | size={row['size_MB']:6.2f}MB")

## 10. Persist experiment summary

In [ ]:
create_results_summary(
    results={
        "fp32_metrics": fp32_metrics,
        "int8_metrics": int8_metrics,
        "fp32_training_results": fp32_training_results,
        "qat_training_results": qat_training_results,
    },
    config=asdict(fp32_cfg),
    output_path=RESULTS_DIR / "experiment_summary.json",
)

## W&B — syncing offline runs

All runs were saved locally with `mode="offline"`. Two run groups are tracked:
- **`fp32-phase3`** — one run per architecture, FP32 training
- **`qat-phase3`** — one run per architecture (excl. `alexnet_se`), QAT fine-tuning

When ready, sync to the W&B dashboard from a terminal:

```bash
wandb sync --sync-all
```